# 82 · Titanic — Ingeniería de features (Experimentos E2 y E3)


Construir la capa **silver** con variables derivadas y  dos experimentos para aislar el efecto de :
- **E2:** mismas familia de modelo que el baseline (regresión logística), pero con *features* enriquecidas.
- **E3:** mismas *features* que E2, pero cambiando la familia del modelo a *Random Forest*.

### Objetivos de aprendizaje
- Derivar variables informativas (`Title`, `FamilySize`, `IsAlone`, `Deck`) con funciones nativas de Spark.
- Imputar por grupo (`Age` por `Title`×`Pclass`) evitando *data leakage*.
- **Persistir los parámetros de imputación como parte del modelo, no de los datos.**
- Contrastar el registro **manual** (E1) con `mlflow.autolog()`.

## 1. Configuración

In [ ]:
import mlflow
from pyspark.sql import functions as F

CATALOGO = "big_data_ii_2025"
ESQUEMA = "spark_examples"
TABLA_BRONZE_TRAIN = f"{CATALOGO}.{ESQUEMA}.titanic_bronze_train"
TABLA_SILVER_TRAIN = f"{CATALOGO}.{ESQUEMA}.titanic_silver_train"
TABLA_PARAMS_IMPUT = f"{CATALOGO}.{ESQUEMA}.titanic_parametros_imputacion"
RUTA_EXPERIMENTO = f"/Volumes/{CATALOGO}/{ESQUEMA}/titanic_mlflow"

mlflow.set_registry_uri("databricks-uc")
# usuario = spark.sql("SELECT current_user()").first()[0]
# mlflow.set_experiment(f"/Users/{usuario}/titanic_mlflow")
mlflow.set_experiment(dfs_tmpdir=f"{RUTA_EXPERIMENTO}")
print("Experimento:", f"{RUTA_EXPERIMENTO}")

## 2. Diseño de las variables derivadas

Construimos, con `regexp_extract` y aritmética de columnas (sin UDFs de Python):

| Variable | Definición |
|---|---|
| `Title` | Título extraído del nombre (`Mr`, `Mrs`, `Miss`, `Master`, `Rare`) |
| `FamilySize` | `SibSp + Parch + 1` |
| `IsAlone` | `1` si `FamilySize == 1`, si no `0` |
| `AgeImputed` | `Age`, imputada con la mediana por (`Title`, `Pclass`) de train |
| `FareImputed` | `Fare`, imputada con la mediana por `Pclass` de train |
| `EmbarkedFill` | `Embarked`, nulos → moda de train (`S`) |
| `Deck` | Primera letra de `Cabin`; nulos → `U` |

In [ ]:
df_bronze = spark.table(TABLA_BRONZE_TRAIN)

# Extraemos el título rraw: el texto entre la coma y el punto ("Braund, Mr. Owen" -> "Mr").
titulos_crudos = (
    df_bronze
    .withColumn("TituloCrudo", F.regexp_extract(F.col("Name"), r",\s*([^\.]+)\.", 1))
    .groupBy("TituloCrudo").count().orderBy(F.desc("count"))
)
n_titulos = titulos_crudos.count()
print(f"Títulos crudos distintos: {n_titulos}")
titulos_crudos.display()

Títulos raw (Mr, Miss, Mrs, Master, Dr, Rev, Col, Major, Mlle, Ms, Mme,
Lady, Sir, Capt, Don, Jonkheer, Countess). La idea es consolidarlos en cinco grupos.

## 3. Función reutilizable de features

`construir_features` recibe los **parámetros de imputación** (calculados solo con train) como
argumentos. Los parámetros de imputación son parte del *modelo*, no de los datos.

In [ ]:
TITULOS_PRINCIPALES = ["Mr", "Mrs", "Miss", "Master"]


def construir_features(df, medianas_age, mediana_fare_pclass, moda_embarked):
    """Aplica todas las transformaciones de la capa silver.

    Parámetros de imputación (calculados sobre train):
      - medianas_age: dict {(Title, Pclass): mediana}
      - mediana_fare_pclass: dict {Pclass: mediana}
      - moda_embarked: str
    """
    # Título crudo.
    d = df.withColumn("TituloCrudo", F.regexp_extract(F.col("Name"), r",\s*([^\.]+)\.", 1))

    # Consolidación de títulos.
    d = d.withColumn(
        "Title",
        F.when(F.col("TituloCrudo").isin(["Mlle", "Ms"]), F.lit("Miss"))
         .when(F.col("TituloCrudo") == "Mme", F.lit("Mrs"))
         .when(F.col("TituloCrudo").isin(TITULOS_PRINCIPALES), F.col("TituloCrudo"))
         .otherwise(F.lit("Rare")),
    )

    # Tamaño de familia y viaja-solo.
    d = d.withColumn("FamilySize", F.col("SibSp") + F.col("Parch") + F.lit(1))
    d = d.withColumn("IsAlone", (F.col("FamilySize") == 1).cast("int"))

    # Imputación de Age por (Title, Pclass) con broadcast de un diccionario.
    # Construimos una expresión CASE encadenada a partir del dict.
    expr_age = F.col("Age")
    mediana_age_global = float(
        sum(medianas_age.values()) / max(len(medianas_age), 1)
    )
    cond = F.col("Age").isNull()
    age_case = F.when(F.lit(False), F.lit(None))  # semilla
    for (titulo, pclass), mediana in medianas_age.items():
        age_case = age_case.when(
            cond & (F.col("Title") == titulo) & (F.col("Pclass") == pclass),
            F.lit(float(mediana)),
        )
    d = d.withColumn(
        "AgeImputed",
        F.when(F.col("Age").isNotNull(), F.col("Age"))
         .otherwise(F.coalesce(age_case, F.lit(mediana_age_global))),
    )

    # Imputación de Fare por Pclass.
    fare_case = F.when(F.lit(False), F.lit(None))
    for pclass, mediana in mediana_fare_pclass.items():
        fare_case = fare_case.when(F.col("Pclass") == pclass, F.lit(float(mediana)))
    d = d.withColumn(
        "FareImputed",
        F.when(F.col("Fare").isNotNull(), F.col("Fare")).otherwise(fare_case),
    )

    # Embarked: nulos -> moda.
    d = d.withColumn(
        "EmbarkedFill",
        F.when(F.col("Embarked").isNull(), F.lit(moda_embarked)).otherwise(F.col("Embarked")),
    )

    # Deck: primera letra de Cabin, nulos -> "U".
    d = d.withColumn(
        "Deck",
        F.when(F.col("Cabin").isNull(), F.lit("U")).otherwise(F.substring(F.col("Cabin"), 1, 1)),
    )

    return d.drop("TituloCrudo")

## 4. Calcular los parámetros de imputacióntelos

Primero necesitamos `Title` para agrupar; lo calculamos con una pasada preliminar. Luego derivamos las
medianas y la moda, y las guardamos en una tabla Delta.

In [ ]:
# Pasada preliminar: solo Title y FamilySize, para poder agrupar.
df_prelim = (
    df_bronze
    .withColumn("TituloCrudo", F.regexp_extract(F.col("Name"), r",\s*([^\.]+)\.", 1))
    .withColumn(
        "Title",
        F.when(F.col("TituloCrudo").isin(["Mlle", "Ms"]), F.lit("Miss"))
         .when(F.col("TituloCrudo") == "Mme", F.lit("Mrs"))
         .when(F.col("TituloCrudo").isin(TITULOS_PRINCIPALES), F.col("TituloCrudo"))
         .otherwise(F.lit("Rare")),
    )
)

# Medianas de Age por (Title, Pclass).
filas_age = (
    df_prelim.filter(F.col("Age").isNotNull())
    .groupBy("Title", "Pclass")
    .agg(F.expr("percentile_approx(Age, 0.5)").alias("mediana"))
    .collect()
)
medianas_age = {(r["Title"], r["Pclass"]): float(r["mediana"]) for r in filas_age}

# Medianas de Fare por Pclass.
filas_fare = (
    df_prelim.filter(F.col("Fare").isNotNull())
    .groupBy("Pclass")
    .agg(F.expr("percentile_approx(Fare, 0.5)").alias("mediana"))
    .collect()
)
mediana_fare_pclass = {int(r["Pclass"]): float(r["mediana"]) for r in filas_fare}

# Moda de Embarked.
moda_embarked = (
    df_prelim.filter(F.col("Embarked").isNotNull())
    .groupBy("Embarked").count().orderBy(F.desc("count")).first()["Embarked"]
)

print("Medianas Age por (Title, Pclass):", medianas_age)
print("Medianas Fare por Pclass:", mediana_fare_pclass)
print("Moda Embarked:", moda_embarked)
assert moda_embarked == "S", f"Se esperaba moda 'S', se obtuvo {moda_embarked}"

In [ ]:
# Persistimos los parámetros como una tabla Delta (clave/valor) para reutilizarlos después
filas_params = []
for (titulo, pclass), mediana in medianas_age.items():
    filas_params.append(("age", f"{titulo}|{pclass}", float(mediana)))
for pclass, mediana in mediana_fare_pclass.items():
    filas_params.append(("fare", str(pclass), float(mediana)))
filas_params.append(("embarked_moda", "moda", None))

df_params = spark.createDataFrame(
    [(t, k, v) for (t, k, v) in filas_params],
    schema="tipo string, clave string, valor double",
)
# Guardamos también la moda como fila con clave textual aparte.
df_params = df_params.unionByName(
    spark.createDataFrame([("embarked_moda_valor", moda_embarked, None)],
                          schema="tipo string, clave string, valor double")
)
df_params.write.mode("overwrite").saveAsTable(TABLA_PARAMS_IMPUT)
print(f"Parámetros de imputación guardados en {TABLA_PARAMS_IMPUT}")

## 5. Construir la capa silver

In [ ]:
df_silver = construir_features(df_bronze, medianas_age, mediana_fare_pclass, moda_embarked)

# Verificación: no deben quedar nulos en las columnas imputadas.
nulos_age = df_silver.filter(F.col("AgeImputed").isNull()).count()
nulos_fare = df_silver.filter(F.col("FareImputed").isNull()).count()
nulos_emb = df_silver.filter(F.col("EmbarkedFill").isNull()).count()
assert nulos_age == 0 and nulos_fare == 0 and nulos_emb == 0, (
    f"Quedan nulos tras imputar: Age={nulos_age}, Fare={nulos_fare}, Embarked={nulos_emb}"
)
print("Sin nulos en AgeImputed, FareImputed ni EmbarkedFill.")

df_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_SILVER_TRAIN)
print(f"Capa silver escrita en {TABLA_SILVER_TRAIN}: {spark.table(TABLA_SILVER_TRAIN).count()} filas")

In [ ]:
df_silver.select(
    "Name", "Title", "FamilySize", "IsAlone", "Age", "AgeImputed",
    "Fare", "FareImputed", "Embarked", "EmbarkedFill", "Deck",
).limit(10).display()

## 6. Experimento E2 — Regresión logística con features enriquecidas

Mismo modelo que el baseline (regresión logística), **features nuevas**. A partir de aquí activamos
`mlflow.autolog()` para que MLflow registre parámetros, métricas y el modelo automáticamente.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator, BinaryClassificationEvaluator,
)

# Split reproducible sobre la capa silver (misma semilla que en 81).
silver = spark.table(TABLA_SILVER_TRAIN)
silver_ent, silver_val = silver.randomSplit([0.8, 0.2], seed=42)

CATEGORICAS = ["Sex", "Title", "EmbarkedFill", "Deck"]
NUMERICAS = ["Pclass", "AgeImputed", "FareImputed", "FamilySize", "IsAlone"]


def etapas_preproceso():
    """Indexado + one-hot de categóricas y ensamblado con numéricas."""
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
        for c in CATEGORICAS
    ]
    encoder = OneHotEncoder(
        inputCols=[f"{c}_idx" for c in CATEGORICAS],
        outputCols=[f"{c}_ohe" for c in CATEGORICAS],
    )
    assembler = VectorAssembler(
        inputCols=NUMERICAS + [f"{c}_ohe" for c in CATEGORICAS],
        outputCol="features",
        handleInvalid="keep",
    )
    return indexers + [encoder, assembler]


def evaluar(pred_df):
    acc = MulticlassClassificationEvaluator(
        labelCol="Survived", predictionCol="prediction", metricName="accuracy").evaluate(pred_df)
    f1 = MulticlassClassificationEvaluator(
        labelCol="Survived", predictionCol="prediction", metricName="f1").evaluate(pred_df)
    auc = BinaryClassificationEvaluator(
        labelCol="Survived", rawPredictionCol="probability", metricName="areaUnderROC").evaluate(pred_df)
    return acc, f1, auc

In [ ]:
# Activamos autolog para modelos de SparkML.
mlflow.pyspark.ml.autolog()

with mlflow.start_run(run_name="E2_logreg_features") as run:
    mlflow.set_tags({"experimento": "E2", "modelo": "LogisticRegression", "notebook": "82"})

    lr = LogisticRegression(featuresCol="features", labelCol="Survived", maxIter=50)
    pipeline_e2 = Pipeline(stages=etapas_preproceso() + [lr])
    modelo_e2 = pipeline_e2.fit(silver_ent)

    pred_val = modelo_e2.transform(silver_val)
    acc_e2, f1_e2, auc_e2 = evaluar(pred_val)

    # autolog ya registró el modelo y muchos params; agregamos nuestras métricas de validación explícitas.
    mlflow.log_metric("accuracy", acc_e2)
    mlflow.log_metric("f1", f1_e2)
    mlflow.log_metric("auc_roc", auc_e2)

print(f"E2 · accuracy={acc_e2:.4f}  f1={f1_e2:.4f}  auc={auc_e2:.4f}")

### Manual (E1) vs autolog (E2)
En E1 escribiste cada `log_param` y `log_metric` a mano. Con `autolog()`, MLflow capturó automáticamente
los hiperparámetros del `Pipeline`, las métricas de entrenamiento y el modelo. El registro manual te da
control total (útil para métricas personalizadas como el *accuracy* de validación); el autolog reduce
código repetitivo. En la práctica se combinan: autolog + algunas métricas manuales, como hicimos aquí.

## 7. Experimento E3 — Random Forest con las MISMAS features

Ahora mantenemos las *features* de E2 constantes y **solo cambiamos el modelo** a *Random Forest*. Es el
espejo del paso E1→E2: allí cambiamos features con modelo fijo; aquí cambiamos modelo con features fijas.

In [ ]:
with mlflow.start_run(run_name="E3_randomforest_features") as run:
    mlflow.set_tags({"experimento": "E3", "modelo": "RandomForest", "notebook": "82"})

    rf = RandomForestClassifier(
        featuresCol="features", labelCol="Survived",
        numTrees=100, maxDepth=5, seed=42,
    )
    pipeline_e3 = Pipeline(stages=etapas_preproceso() + [rf])
    modelo_e3 = pipeline_e3.fit(silver_ent)

    pred_val = modelo_e3.transform(silver_val)
    acc_e3, f1_e3, auc_e3 = evaluar(pred_val)

    mlflow.log_metric("accuracy", acc_e3)
    mlflow.log_metric("f1", f1_e3)
    mlflow.log_metric("auc_roc", auc_e3)

print(f"E3 · accuracy={acc_e3:.4f}  f1={f1_e3:.4f}  auc={auc_e3:.4f}")

## 8. Compara E1, E2 y E3 en la interfaz

1. Abre **Experiments → `/Users/<tu_usuario>/titanic_mlflow`**.
2. Marca las casillas de los *runs* `E1_baseline_logreg`, `E2_logreg_features` y `E3_randomforest_features`.
3. Pulsa **Compare**. Revisa la tabla de parámetros lado a lado y prueba el **Parallel Coordinates Plot**.
4. En la pestaña **Chart** del experimento agrega un gráfico de barras con la métrica `accuracy` por *run*.

Interpreta: normalmente el salto grande viene de **E1→E2** (las *features* importan mucho), mientras que
**E2→E3** aporta una mejora más modesta. Eso muestra que en este problema invertir en *features* rinde
tanto o más que cambiar de algoritmo.